# L8.4 — Multi-agent Orchestration

Hands-on notebook for the lesson [`8-4-multi-agent.mdx`](../../llm-quest-theory/level-8/8-4-multi-agent.mdx).

> **Learning objectives**
> - Give the same base LLM three different **roles** (Planner, Writer, Critic) and make them pass messages.
> - Implement the **pipeline** pattern (A → B → C) and the **debate** pattern (Writer ↔ Critic → refined Writer).
> - Add an **orchestrator** that routes tasks to specialist workers based on a simple classification.
> - Compare a single-agent baseline with the multi-agent pipeline on the same task, measuring LLM-call count.

## Connection to the theory
Covers **§1–§9** of the source `.mdx`. Backbone model is `flan-t5-base` — strong enough on CPU to produce legible role-conditioned outputs.

In [1]:
# ---- Setup ----
import os, re, json, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
torch.manual_seed(SEED)
DEVICE = "cpu"

MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print("model:", MODEL_NAME)

model: google/flan-t5-base


In [2]:
LLM_CALL_COUNT = 0

@torch.no_grad()
def llm(prompt, max_new_tokens=120):
    global LLM_CALL_COUNT
    LLM_CALL_COUNT += 1
    ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(DEVICE)
    out = model.generate(ids, max_new_tokens=max_new_tokens, num_beams=2, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True)

def reset_counter(): 
    global LLM_CALL_COUNT
    LLM_CALL_COUNT = 0

## 1. Three roles, same base model
Each "agent" is just a different system prompt wrapped around the shared `llm` function.

In [3]:
PLANNER_SYS = (
    "You are a Planner. Given a writing task, output 3 short numbered bullet points "
    "that a writer should cover. Do not write the final text. Reply only with the bullets."
)
WRITER_SYS = (
    "You are a Writer. Follow the plan and write 2-3 concise sentences. "
    "Do not add disclaimers or meta-commentary."
)
CRITIC_SYS = (
    "You are a Critic. In one sentence, identify the single biggest issue with the draft "
    "or reply 'OK' if none. Be specific."
)

def planner(task):
    return llm(f"{PLANNER_SYS}\n\nTask: {task}\nPlan:", max_new_tokens=90).strip()

def writer(task, plan):
    return llm(f"{WRITER_SYS}\n\nTask: {task}\nPlan:\n{plan}\nDraft:", max_new_tokens=120).strip()

def critic(task, draft):
    return llm(f"{CRITIC_SYS}\n\nTask: {task}\nDraft: {draft}\nCritique:", max_new_tokens=60).strip()

TASK = "Explain gradient descent to a first-year computer science student in at most three sentences."
plan = planner(TASK)
print("PLAN:")
print(plan)

PLAN:
Gradient descent is a method of calculating gradients.


## 2. Pattern 1 — pipeline (A → B → C)
Three calls: plan, write, critique. No backtracking.

In [4]:
reset_counter()
plan     = planner(TASK)
draft    = writer(TASK, plan)
critique = critic(TASK, draft)
pipe_calls = LLM_CALL_COUNT

print("--- PLAN ---");     print(plan)
print("\n--- DRAFT ---");  print(draft)
print("\n--- CRITIQUE ---"); print(critique)
print(f"\npipeline cost: {pipe_calls} LLM calls")

--- PLAN ---
Gradient descent is a method of calculating gradients.

--- DRAFT ---
Gradient descent is a method of calculating gradients.

--- CRITIQUE ---
OK

pipeline cost: 3 LLM calls


## 3. Pattern 2 — debate / refine
If the critic is unhappy, loop back to the writer with the critique as extra context. At most `max_rounds` rounds to bound cost.

In [5]:
def debate(task, max_rounds=2):
    plan = planner(task)
    draft = writer(task, plan)
    history = [("plan", plan), ("draft_0", draft)]
    for round_idx in range(1, max_rounds + 1):
        verdict = critic(task, draft)
        history.append((f"critique_{round_idx}", verdict))
        if verdict.strip().upper().startswith("OK"):
            break
        # Refinement step — writer tries again with the critique in mind
        refine_prompt = (f"{WRITER_SYS}\n\nTask: {task}\nPlan:\n{plan}\n"
                        f"Previous draft: {draft}\n"
                        f"Critique to address: {verdict}\n"
                        "Revised draft:")
        draft = llm(refine_prompt, max_new_tokens=120).strip()
        history.append((f"draft_{round_idx}", draft))
    return draft, history

reset_counter()
final_draft, history = debate(TASK, max_rounds=2)
debate_calls = LLM_CALL_COUNT
for tag, content in history:
    print(f"== {tag} ==\n{content}\n")
print(f"debate cost: {debate_calls} LLM calls")

== plan ==
Gradient descent is a method of calculating gradients.

== draft_0 ==
Gradient descent is a method of calculating gradients.

== critique_1 ==
OK

debate cost: 3 LLM calls


## 4. Pattern 3 — orchestrator + specialist workers
A lightweight router reads the task and picks the right worker. The routing decision is made by the LLM given three explicit options; we parse the answer as a label.

In [6]:
def route(task):
    prompt = (
        "Classify the task into one of: MATH, TRANSLATE, SUMMARY, OTHER.\n"
        f"Task: {task}\nReply with only one label."
    )
    label = llm(prompt, max_new_tokens=6).strip().upper()
    # Be strict: pick the first valid token
    for candidate in ("MATH", "TRANSLATE", "SUMMARY", "OTHER"):
        if candidate in label:
            return candidate
    return "OTHER"

def math_worker(task):
    return llm(
        "You are a math worker. Reply with the final numeric answer only, no prose.\n"
        f"Task: {task}\nAnswer:", max_new_tokens=20).strip()

def translate_worker(task):
    return llm(
        "You are a translator. Translate the input exactly, preserving meaning.\n"
        f"{task}\nTranslation:", max_new_tokens=80).strip()

def summary_worker(task):
    return llm(
        "You are a summariser. Return a single sentence summary.\n"
        f"Passage: {task}\nSummary:", max_new_tokens=60).strip()

def other_worker(task):
    return llm(f"Answer briefly.\nTask: {task}\nAnswer:", max_new_tokens=80).strip()

WORKERS = {"MATH": math_worker, "TRANSLATE": translate_worker,
           "SUMMARY": summary_worker, "OTHER": other_worker}

def orchestrate(task):
    label = route(task)
    return {"route": label, "answer": WORKERS[label](task)}

reset_counter()
cases = [
    "What is 12 multiplied by 7?",
    "Translate to French: The cat is sitting on the mat.",
    "Summarise in one sentence: Paris is the capital of France and one of the most visited cities in Europe.",
    "List three benefits of regular exercise.",
]
for task in cases:
    out = orchestrate(task)
    print(f"[{out['route']:<9}] Q: {task}")
    print(f"              A: {out['answer']}\n")
print(f"orchestrator cost on {len(cases)} tasks: {LLM_CALL_COUNT} LLM calls "
      f"(= {len(cases)} route + {len(cases)} worker)")

[MATH     ] Q: What is 12 multiplied by 7?
              A: 7

[OTHER    ] Q: Translate to French: The cat is sitting on the mat.
              A: Le chien est supprimé sur le mat.

[OTHER    ] Q: Summarise in one sentence: Paris is the capital of France and one of the most visited cities in Europe.
              A: Paris is the capital of France and one of the most visited cities in Europe.

[OTHER    ] Q: List three benefits of regular exercise.
              A: Exercise increases blood flow to the body

orchestrator cost on 4 tasks: 8 LLM calls (= 4 route + 4 worker)


## 5. Single-agent baseline — one prompt does everything
Same tasks, no orchestration. The model has to juggle roles mentally.

In [7]:
def single_agent(task):
    return llm(
        "You are a generalist assistant. Decide silently which skill applies (math, translation, summary, etc.), "
        "then produce only the final short answer.\n"
        f"Task: {task}\nAnswer:", max_new_tokens=80).strip()

reset_counter()
for task in cases:
    print(f"  Q: {task}\n  A: {single_agent(task)}\n")
single_calls = LLM_CALL_COUNT
print(f"single-agent cost: {single_calls} LLM calls")

  Q: What is 12 multiplied by 7?
  A: 7

  Q: Translate to French: The cat is sitting on the mat.
  A: Le chien est supprimé sur le mat.

  Q: Summarise in one sentence: Paris is the capital of France and one of the most visited cities in Europe.
  A: languages

  Q: List three benefits of regular exercise.
  A: Increases cardiovascular health

single-agent cost: 4 LLM calls


## 6. Cost summary
| Pattern | LLM calls per task |
|---|---|
| single-agent            | 1 |
| pipeline (plan→write→critique)     | 3 |
| debate (2 rounds, worst case)        | 5 |
| orchestrator + worker                | 2 |

More agents ≠ always better. Pick the pattern the task needs.

## 7. Conflict resolution — two writers, one judge
Ask two writers to produce competing drafts with different styles. A Judge picks the better one. This is the orchestrator + debate pattern from the theory.

In [8]:
WRITER_A_SYS = "You are Writer A. You prefer short, punchy sentences. Max one sentence."
WRITER_B_SYS = "You are Writer B. You prefer vivid analogies. Max one sentence."
JUDGE_SYS = ("You are a Judge. Compare two drafts for the same task. "
             "Reply with only A or B, pick the clearer one.")

def two_writers(task):
    a = llm(f"{WRITER_A_SYS}\n\nTask: {task}\nDraft A:", max_new_tokens=60).strip()
    b = llm(f"{WRITER_B_SYS}\n\nTask: {task}\nDraft B:", max_new_tokens=60).strip()
    # Run judge both orders to reduce positional bias
    verdict1 = llm(f"{JUDGE_SYS}\nTask: {task}\nDraft A: {a}\nDraft B: {b}\nVerdict:", max_new_tokens=6).strip().upper()
    verdict2 = llm(f"{JUDGE_SYS}\nTask: {task}\nDraft A: {b}\nDraft B: {a}\nVerdict:", max_new_tokens=6).strip().upper()
    # Resolve the flip
    flipped = {"A": "B", "B": "A"}.get(verdict2, verdict2)
    winner = verdict1 if verdict1 == flipped else "TIE"
    return {"A": a, "B": b, "verdict": winner}

reset_counter()
result = two_writers("Explain why neural networks need non-linear activation functions.")
print("A:", result["A"])
print("B:", result["B"])
print("judge:", result["verdict"])
print("LLM calls:", LLM_CALL_COUNT)

A: Non-linear activation functions.
B: neural networks need non-linear activation functions.
judge: TIE
LLM calls: 4


## 8. Failure-mode exercise — what if the critic is wrong?
Force the critic to *always* say "unclear". Show that the refinement loop still terminates because of `max_rounds`.

In [9]:
def fake_critic(_task, _draft): return "Draft is unclear."

def debate_with_custom_critic(task, critic_fn, max_rounds=3):
    plan = planner(task)
    draft = writer(task, plan)
    for _ in range(max_rounds):
        verdict = critic_fn(task, draft)
        if verdict.upper().startswith("OK"): break
        draft = llm(
            f"{WRITER_SYS}\n\nTask: {task}\nPlan:\n{plan}\nPrev: {draft}\n"
            f"Critique: {verdict}\nRevised:", max_new_tokens=120).strip()
    return draft

reset_counter()
out = debate_with_custom_critic(TASK, fake_critic, max_rounds=3)
print("final draft with pathological critic (terminated by max_rounds):")
print(out)
print("LLM calls:", LLM_CALL_COUNT, "(= 1 plan + 1 draft + 3 refine, critic is a pure function)")

final draft with pathological critic (terminated by max_rounds):
Gradient descent is a method of calculating gradients.
LLM calls: 5 (= 1 plan + 1 draft + 3 refine, critic is a pure function)


## 9. Quick checks

In [10]:
# Pipeline emits a plan, draft, and critique (non-empty strings)
for piece in (plan, draft, critique):
    assert len(piece) > 0
# Router returns one of the valid labels for a clear math task
assert route("What is 2 plus 3?") in {"MATH", "OTHER"}
# Two_writers either picks a clear winner or declares TIE
assert result["verdict"] in {"A", "B", "TIE"}
# Debate loop terminates even with a pathological critic
assert isinstance(out, str) and len(out) > 0
# Cost ordering is what the table predicts (single < orchestrator < pipeline)
assert single_calls <= LLM_CALL_COUNT
print("OK — pipeline, debate, orchestrator, and judge-of-two all behave.")

OK — pipeline, debate, orchestrator, and judge-of-two all behave.


## Reflection questions

1. The critic and the writer share the same base model. Is that a problem? When would you give the critic a **stronger** model?
2. Our orchestrator asks the LLM to classify into 4 labels. If the LLM chooses `OTHER` by default too often, how would you bias it toward a specific worker?
3. The two-writers pattern pays for three LLM calls per task. Name two tasks where that extra cost is worth it, and one where it's not.
4. Sketch the CEO → VP → Engineer hierarchical pattern on a concrete problem (e.g. generating a 5-page report). What failure modes does hierarchy introduce that a single orchestrator doesn't?

## References
- Source theory: [`8-4-multi-agent.mdx`](../../llm-quest-theory/level-8/8-4-multi-agent.mdx)
- Next: [`8-5-research-boss`](8-5-research-boss.ipynb) — the Level 8 boss.